# 02 - Treino LoRA

Prepara o dataset no formato que o script pede e treina 2 configs pra comparar.

## 1. Clonar o repo

In [ ]:
!git clone https://github.com/lramosc1512/atelie-generativo-LeonidIA.git
%cd atelie-generativo-LeonidIA
!ls dados/imagens | wc -l
!head -3 dados/legendas.txt

## 2. Instalar dependencias e clonar o script oficial

In [ ]:
!pip -q install transformers accelerate peft datasets
%cd /content
!git clone https://github.com/huggingface/diffusers
%cd diffusers
!pip -q install -e .
%cd examples/text_to_image
!pip -q install -r requirements.txt
!pip -q install -U torchao

## 3. Login HF

In [ ]:
from huggingface_hub import login
login()

## 4. Converter dataset pro formato imagefolder

O script pede metadata.jsonl (file_name + text), não lê fontes.csv/legendas.txt direto.

In [ ]:
import json
import shutil
from pathlib import Path

ORIGEM_IMAGENS = Path("/content/atelie-generativo-LeonidIA/dados/imagens")
LEGENDAS_TXT = Path("/content/atelie-generativo-LeonidIA/dados/legendas.txt")
PASTA_DATASET = Path("/content/dataset_estilo_brasilia")

PASTA_DATASET.mkdir(parents=True, exist_ok=True)

linhas_metadata = []
with open(LEGENDAS_TXT, "r", encoding="utf-8") as f:
    for linha in f:
        linha = linha.strip()
        if not linha:
            continue
        arquivo, legenda = linha.split(",", 1)
        origem = ORIGEM_IMAGENS / arquivo
        destino = PASTA_DATASET / arquivo
        shutil.copy(origem, destino)
        linhas_metadata.append({"file_name": arquivo, "text": legenda.strip()})

with open(PASTA_DATASET / "metadata.jsonl", "w", encoding="utf-8") as f:
    for item in linhas_metadata:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"{len(linhas_metadata)} imagens copiadas e metadata.jsonl gerado em {PASTA_DATASET}")

## 4.5 Montar o Drive

Assim os checkpoints salvam direto lá, não em /content (que some se a sessão cair).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs("/content/drive/MyDrive/atelie_generativo", exist_ok=True)
print("Drive montado. Checkpoints serao salvos em /content/drive/MyDrive/atelie_generativo/")

## 5. Config 1 - rank=4, lr=1e-4, 1000 steps

Rank baixo + lr mais alto: treina mais rápido, menos risco de overfit com dataset pequeno.

In [ ]:
%cd /content/diffusers/examples/text_to_image

!accelerate launch train_text_to_image_lora.py \
  --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
  --train_data_dir="/content/dataset_estilo_brasilia" \
  --resolution=512 --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --max_train_steps=1000 \
  --learning_rate=1e-4 --lr_scheduler="cosine" \
  --rank=4 \
  --mixed_precision="fp16" \
  --checkpointing_steps=250 \
  --validation_prompt="estilo_brasilia, fachada curva de concreto branco refletida em espelho d'agua ao entardecer" \
  --output_dir="/content/drive/MyDrive/atelie_generativo/lora_config1_rank4" \
  --push_to_hub --hub_model_id="lramosc1512/lora-estilo-brasilia-config1"

## 6. Config 2 - rank=8, lr=5e-5, 1500 steps

Rank maior captura mais detalhe, compensado com lr mais conservador. Rodar só depois da config 1 terminar.

In [ ]:
%cd /content/diffusers/examples/text_to_image

!accelerate launch train_text_to_image_lora.py \
  --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
  --train_data_dir="/content/dataset_estilo_brasilia" \
  --resolution=512 --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --max_train_steps=1500 \
  --learning_rate=5e-5 --lr_scheduler="cosine" \
  --rank=8 \
  --mixed_precision="fp16" \
  --checkpointing_steps=250 \
  --validation_prompt="estilo_brasilia, fachada curva de concreto branco refletida em espelho d'agua ao entardecer" \
  --output_dir="/content/drive/MyDrive/atelie_generativo/lora_config2_rank8" \
  --push_to_hub --hub_model_id="lramosc1512/lora-estilo-brasilia-config2"

## 7. Imagens de validação

Geradas a cada 250 passos com o validation_prompt - dá pra ver a evolução do estilo nos logs acima. Vai servir pra Etapa 3.

## 8. Conferir os checkpoints no Drive

In [ ]:
!ls -la "/content/drive/MyDrive/atelie_generativo/lora_config1_rank4"
!ls -la "/content/drive/MyDrive/atelie_generativo/lora_config2_rank8"

## Depois disso

- Confirmar os 2 modelos publicados no HF
- Preencher model card de cada um
- Guardar as imagens de validação pra grade comparativa da Etapa 3

```bash
git add notebooks/02_treino_lora.ipynb
git commit -m "notebook treino lora"
git push
```